In [109]:
import pandas as pd
import kagglehub
import os
import seaborn as sns
import matplotlib.pyplot as plt

pd.options.display.max_rows = 4000


In [110]:
urban_path = kagglehub.dataset_download("vellis1/us-cities-urban-connectivity")
yelp_path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

In [111]:
files = os.listdir(urban_path)
csv_file = [f for f in files if f.endswith('.csv')][0]
urban_df = pd.read_csv(os.path.join(urban_path, csv_file))
yelp_df = pd.read_json(os.path.join(yelp_path, "yelp_academic_dataset_business.json"), lines=True)

In [112]:
MAJOR_RULES = {
    "Food & Dining": [
        "restaurant","food","cafe","coffee","tea","bar","bakery","dessert","ice cream",
        "pizza","burger","sandwich","noodle","ramen","sushi","mexican","italian","korean",
        "thai","indian","chinese","japanese","mediterranean","halal","vegan","vegetarian",
        "brunch","breakfast","deli","seafood","steakhouse","bbq","brew","wine","pub",
        "grocery","market","butcher","donut","bagel","taco","poke","buffet","food truck",
        "american (traditional)","american (new)","salad","chicken wings","cajun","creole",
        "southern","diners","latin american","soup","vietnamese","asian fusion",
        "middle eastern","gluten-free","tex-mex","greek","french","cuban","fish & chips",
        "dim sum","creperies","hawaiian","falafel","african","puerto rican","turkish",
        "german","cantonese","hot pot","filipino","brazilian","ethiopian","kosher",
        "pakistani","irish","szechuan","lebanese","kebab","brasseries","pan asian",
        "empanadas","persian","iranian","teppanyaki","malaysian","fondue","izakaya",
        "shanghainese","mongolian","himalayan","nepalese","ukrainian","egyptian",
        "singaporean","burmese","armenian","syrian","scandinavian","australian",
        "bangladeshi","sicilian","senegalese","haitian","trinidadian","iberian",
        "hungarian","somali","sardinian","georgian","sri lankan","guamanian",
        "serbo croatian","czech","hainan","israeli","fuzhou","south african",
        "colombian","peruvian","venezuelan","salvadoran","dominican","russian",
        "honduran","laotian","argentine","belgian","portuguese","british","moroccan",
        "afghan","arabic","polish","cambodian","indonesian","basque","taiwanese",
        "smokehouse","pretzels","waffles","shaved ice","acai bowls","gelato",
        "custom cakes","cupcakes","fruits & veggies","wraps","kombucha",
        "herbs & spices","olive oil","honey","pancakes","beer hall","poutineries",
        "bistros","street vendors","personal chefs","distilleries","cideries",
        "meaderies","speakeasies","beer","cucina campana","tuscan","austrian",
        "ukrainian", "bakeries"
    ],

    "Shopping & Retail": [
        "store","shop","shopping","fashion","clothing","jewelry","gift","book","electronics",
        "furniture","thrift","antique","cosmetic","supply","wholesale","mall","marketplace",
        "shoe","accessories","toy","department","boutique",
        "home decor","sporting goods","mags","used","vintage & consignment",
        "mobile phones","building supplies","kitchen & bath","eyewear & opticians",
        "mattresses","watches","lighting fixtures","outdoor gear","guns & ammo",
        "leather goods","vinyl records","lingerie","costumes","formal wear",
        "motorcycle gear","hats","perfume","luggage","swimwear","knitting supplies",
        "customized merchandise","wigs","religious items","uniforms","kitchen supplies",
        "office equipment","rugs","shades & blinds","tableware","military surplus",
        "hunting & fishing supplies","gemstones & minerals","gold buyers",
        "diamond buyers","firewood","safety equipment","vitamins & supplements",
        "computers","signmaking","framing","grilling equipment"
    ],

    "Beauty & Personal Care": [
        "salon","spa","hair","nail","wax","tattoo","piercing","massage","skin","makeup",
        "barber","tanning","eyelash","threading","facial",
        "blow dry/out services","eyebrow services","sugaring","acne treatment",
        "aestheticians","estheticians"
    ],

    "Health & Medical": [
        "doctor","dentist","hospital","medical","clinic","therapy","chiropractor",
        "optometrist","psychologist","psychiatrist","surgeon","pharmacy","rehab",
        "urgent care","health","wellness","nutrition",
        "orthodontists","endodontists","periodontists","family practice",
        "acupuncture","reflexology","weight loss centers","diagnostic services",
        "dermatologists","ophthalmologists","internal medicine","obstetricians",
        "gynecologists","pediatricians","orthopedists","naturopathic","holistic",
        "reiki","laboratory testing","pain management","laser eye surgery","lasik",
        "allergists","gastroenterologist","neurologist","podiatrists","ear nose & throat",
        "radiologists","urologists","fertility","endocrinologists","oncologist",
        "sleep specialists","osteopathic physicians","audiologist","prosthodontists",
        "nurse practitioner","addiction medicine","dental hygienists","dietitians",
        "hearing aid providers","pulmonologist","speech therapists","vascular medicine",
        "emergency rooms","emergency medicine","iv hydration","colonics",
        "alternative medicine","meditation centers","tui na","tai chi","qi gong",
        "ayurveda","preventive medicine","concierge medicine","lactation services",
        "midwives","hospice","skilled nursing","assisted living facilities",
        "diagnostic imaging","teeth whitening","orthotics","immunodermatologists",
        "neurotologists","neuropathologists","proctologists","nephrologists",
        "phlebologists","hepatologists","infectious disease specialists",
        "pathologists","retina specialists","anesthesiologists","rheumatologists",
        "gerontologists","behavior analysts","sex therapists",
        "mobility equipment sales & services","prosthetics"
    ],

    "Automotive": [
        "auto","car","vehicle","tire","oil change","repair","dealer","towing","smog",
        "body shop","transmission","glass","detailing",
        "gas stations","roadside assistance","ev charging stations"
    ],

    "Home & Local Services": [
        "plumbing","electrician","hvac","roof","contractor","cleaning","locksmith",
        "moving","landscaping","flooring","installation","repair","maintenance","handyman",
        "appliance","home service","garden",
        "local services","movers","sewing & alterations","printing services",
        "shipping centers","pest control","painters","tree services",
        "junk removal & hauling","damage restoration","masonry/concrete",
        "packing services","pool & hot tub service","irrigation","window washing",
        "landscape architects","pool cleaners","gutter services","pressure washers",
        "garage door services","fences & gates","home window tinting","cabinetry",
        "tiling","laundry services","laundromat","refinishing services",
        "grout services","demolition services","stucco services","snow removal",
        "siding","decks & railing","chimney sweeps","fireplace services",
        "hydro-jetting","septic services","tv mounting","shutters","patio coverings",
        "home inspectors","home organization","home staging","awnings",
        "fire protection services","environmental abatement","screen printing",
        "interior design","florists","floral designers","swimming pools",
        "hot tub & pool","home energy auditors","well drilling","excavation services",
        "backflow services","wallpapering","powder coating","grill services",
        "wildlife control","home developers","interlock systems",
        "sheds & outdoor storage","security systems","security services",
        "environmental testing"
    ],

    "Real Estate & Housing": [
        "real estate","apartment","property","mortgage","housing","leasing",
        "condominiums","self storage","homeowner association"
    ],

    "Professional & Financial Services": [
        "law","legal","account","insurance","tax","consult","financial","bank",
        "notary","payday","broker","business service",
        "professional services","general litigation","investing",
        "check cashing/pay-day loans","title loans","installment loans",
        "debt relief services","payroll services","employment agencies",
        "advertising","graphic design","web design","appraisal services",
        "registration services","personal assistants","wills","trusts","& probates",
        "business financing","process servers","editorial services",
        "translation services","software development","billing services",
        "data recovery","internet service providers","utilities",
        "electricity suppliers","talent agencies","structural engineers"
    ],

    "Entertainment & Recreation": [
        "gym","fitness","park","museum","art","music","theater","cinema","club",
        "karaoke","festival","sports","golf","yoga","dance","arcade","game",
        "recreation","bowling","casino",
        "nightlife","active life","event planning & services","lounges",
        "trainers","boot camps","pilates","jazz & blues","kids activities",
        "pool halls","boxing","stadiums & arenas","skating rinks","hiking",
        "boating","kickboxing","tennis","mountain biking","climbing",
        "brazilian jiu-jitsu","horseback riding","fishing","gun/rifle ranges",
        "rafting/kayaking","axe throwing","paint & sip","challenge courses",
        "diving","scuba diving","rock climbing","laser tag","paintball",
        "virtual reality centers","archery","skydiving","surfing","tubing",
        "hot air balloons","batting cages","sailing","free diving",
        "adult entertainment","adult","haunted houses","scavenger hunts",
        "jet skis","soccer","basketball courts","baseball fields","marinas",
        "opera & ballet","playgrounds","summer camps","day camps",
        "indoor playcentre","aquariums","zoos","landmarks & historical buildings",
        "local flavor","libraries","beaches","lakes","campgrounds",
        "horse racing","race tracks","djs","pool & billiards","eatertainment",
        "pumpkin patches","attraction farms","bike sharing","bicycle paths",
        "pickleball","bocce ball","badminton","skiing","sledding",
        "outdoor movies","bingo halls","rodeo","airsoft","cannabis dispensaries",
        "cannabis collective","bikes","swimming pools","snowboarding",
        "hang gliding","ziplining","snorkeling","trivia hosts",
        "recording & rehearsal studios","video/film production","magicians",
        "face painting","clowns","wildlife hunting ranges","metal detector services"
    ],

    "Education": [
        "school","college","university","tutor","training","education","class","lesson",
        "test preparation"
    ],

    "Travel & Transportation": [
        "hotel","travel","tour","airport","taxi","transport","rental","shuttle",
        "airline","resort","vacation",
        "limos","guest houses","hostels","train stations","buses","bus stations",
        "ferries","pedicabs","valet services","luggage storage","rest stops",
        "visitor centers","trains","flight instruction"
    ],

    "Pets": [
        "pet","veterinarian","animal","dog","cat","groomer","boarding",
        "aquarium services"
    ],

    "Community & Public Services": [
        "community service/non-profit","religious organizations","churches",
        "cultural center","community centers","funeral services & cemeteries",
        "cremation services","mortuary services","post offices","courthouses",
        "donation center","recycling center","blood & plasma donation centers",
        "synagogues","buddhist temples","hindu temples","mosques",
        "homeless shelters","jails & prisons","municipality","embassy",
        "crisis pregnancy centers","faith-based crisis pregnancy centers",
        "adoption services","hospice"
    ],

    "Events & Weddings": [
        "wedding planning","bridal","photographers","session photography",
        "event photography","boudoir photography","videographers",
        "wedding chapels","officiants","balloon services","holiday decorations",
        "holiday decorating services","christmas trees","paint-your-own pottery",
        "screen printing/t-shirt printing","engraving","calligraphy",
        "digitizing services","customized merchandise"
    ]
}

def map_major_category(category):
  if pd.isna(category):
    return "Other"
  c = category.lower()

  for major, keywords in MAJOR_RULES.items():
    for k in keywords:
      if k in c:
        return major
  return "Other"

In [113]:
print(f"Urban DF Columns: {urban_df.columns.to_list()}")
print(f"Yelp DF Columns: {yelp_df.columns.to_list()}")

Urban DF Columns: ['Unnamed: 0', 'Place_name', 'City', 'State', 'Walk Score', 'Transit Score', 'Bike Score', 'Population_2021', 'Population_2022_Census', 'City_Population_Stats_adjusted_land_area__acres_', 'City_Population_Stats_density__people_acre_', 'City_Population_Stats_density_classification', 'City_Population_Stats_2000_population', 'City_Population_Stats_population_growth', 'City_Population_Stats_population_growth_classification', 'Parkland_Stats_by_City_total_acres', 'Parkland_Stats_by_City__natural', 'Parkland_Stats_by_City__designed', 'Parkland_Stats_by_City_total_park_units', 'Parkland_Stats_by_City_parks_as__city_area', 'Walkable_Park_Access_all_residents', 'Walkable_Park_Access_black', 'Walkable_Park_Access_hispanic_latinx', 'Walkable_Park_Access_asian', 'Walkable_Park_Access_other_race', 'Walkable_Park_Access_multiple_races', 'Walkable_Park_Access_pacific_islander', 'Walkable_Park_Access_american_indian_alaska_native', 'Walkable_Park_Access_all_people_of_color', 'Walkabl

In [114]:
# Data Cleaning

urban_df = urban_df.rename(columns={
    "State": "state",
    "City": 'city'
})

combined = pd.merge(left=urban_df, right=yelp_df, on=["state"], how="inner")
combined = combined.drop(columns=["Unnamed: 0"])
combined['categories'] = combined['categories'].str.split(", ")
combined = combined.explode('categories')
combined["major_category"] = combined["categories"].apply(map_major_category)
combined['Travel_Score'] = (combined['Walk Score'] + combined['Transit Score'] + combined['Bike Score']) / 3
combined = combined[combined['is_open'] == 1]

to_drop = ['Population_2021',
    'Population_2022_Census',
    'City_Population_Stats_adjusted_land_area__acres_',
    'City_Population_Stats_density__people_acre_',
    'City_Population_Stats_density_classification',
    'City_Population_Stats_2000_population',
    'City_Population_Stats_population_growth',
    'City_Population_Stats_population_growth_classification',
    'Parkland_Stats_by_City_total_acres',
    'Parkland_Stats_by_City__natural',
    'Parkland_Stats_by_City__designed',
    'Parkland_Stats_by_City_total_park_units',
    'Parkland_Stats_by_City_parks_as__city_area',
    'Walkable_Park_Access_all_residents',
    'Walkable_Park_Access_black',
    'Walkable_Park_Access_hispanic_latinx',
    'Walkable_Park_Access_asian',
    'Walkable_Park_Access_other_race',
    'Walkable_Park_Access_multiple_races',
    'Walkable_Park_Access_pacific_islander',
    'Walkable_Park_Access_american_indian_alaska_native',
    'Walkable_Park_Access_all_people_of_color',
    'Walkable_Park_Access_white',
    'Walkable_Park_Access_low__75_city_income_',
    'Walkable_Park_Access_middle',
    'Walkable_Park_Access_high__125_city_median_income_',
    'Walkable_Park_Access_children__u19_',
    'Walkable_Park_Access_adults__19_64_',
    'Walkable_Park_Access_seniors__65_',
    'Distribution_of_Park_Space_low_income',
    'Distribution_of_Park_Space_high_income',
    'Distribution_of_Park_Space_black',
    'Distribution_of_Park_Space_hispanic_latnix',
    'Distribution_of_Park_Space_asian',
    'Distribution_of_Park_Space_other_race',
    'Distribution_of_Park_Space_multiple_races',
    'Distribution_of_Park_Space_pacific_islander',
    'Distribution_of_Park_Space_american_indian_alaska_native',
    'Distribution_of_Park_Space_white',
    'Distribution_of_Park_Space_neighborhoods_of_color',
    'Number_of_Fields_and_Diamonds',
    'Number_of_Courts',
    'Number_of_Tennis_courts',
    'Number_of_Pickleball_courts',
    'Number_of_Pickleball_courts_overlay_tennis',
    'Number_of_Standalone_Pickleball',
    'Number_of_Volleyball_nets',
    'Number_of_basketball_hoops_at_park_sites',
    'Number_of_basketball_hoops_at_community_schoolyards',
    'Number_of_Beaches',
    'Number_of_Community_Garden_Sites',
    'Number_of_Community_Garden_Plots',
    'Number_of_Cooling_Centers',
    'Number_of_Dog_Parks',
    'Number_of_Drinking_fountains',
    'Number_of_Playgrounds',
    'Number_of_Rec_Centers',
    'Number_of_senior_centers',
    'Number_of_Restrooms_Freestanding_permanent',
    'Number_of_Restrooms_In_park_buildings',
    'Number_of_Restrooms_Semi_permanent',
    'Number_of_Skate_parks',
    'Number_of_Splashpads',
    'Number_of_Swimming_pools',
    'Number_of_Disc_Golf_Courses',
    'Trail_Miles_Improved_Trails',
    'Trail_Miles_Nature_trails',
    'Number_of_Tracks_At_Park_Sites',
    'Number_of_Tracks_At_community_schoolyards',
    'Number_of_Exercise_Zones',
    'Miles_of_car_free_roadways_in_parks',
    'Permanent_car_free_roadways', "address", "business_id", "is_open", 'attributes', 'hours', 'review_count', 'stars']

combined = combined.drop(columns=to_drop)



In [115]:
numeric_list = combined.select_dtypes(include=['int64', 'float64']).columns.tolist()

food_dining_df = combined[combined['major_category'] == "Food & Dining"]

corr_matrix = food_dining_df[numeric_list].corr()
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[abs(corr_pairs) < 1].drop_duplicates()
print(corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index))

Walk Score     Travel_Score     0.966952
Transit Score  Travel_Score     0.959850
Walk Score     Transit Score    0.922867
Bike Score     Travel_Score     0.778172
Walk Score     Bike Score       0.646775
Transit Score  Bike Score       0.629231
               longitude        0.396561
               latitude         0.370201
Walk Score     longitude        0.364126
longitude      Travel_Score     0.332534
latitude       Travel_Score     0.231902
Walk Score     latitude         0.209884
latitude       longitude       -0.142895
Bike Score     longitude        0.040285
               latitude         0.004483
dtype: float64


In [116]:
shopping_df = combined[combined['major_category'] == "Shopping & Retail"]
corr_matrix = shopping_df[numeric_list].corr()
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[abs(corr_pairs) < 1].drop_duplicates()
print(corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index))

Walk Score     Travel_Score     0.966146
Transit Score  Travel_Score     0.956373
Walk Score     Transit Score    0.918946
Bike Score     Travel_Score     0.766203
Walk Score     Bike Score       0.629856
Transit Score  Bike Score       0.605812
               longitude        0.382260
Walk Score     longitude        0.360614
longitude      Travel_Score     0.327105
Transit Score  latitude         0.309052
latitude       longitude       -0.217817
               Travel_Score     0.168168
Walk Score     latitude         0.143147
Bike Score     latitude        -0.038343
               longitude        0.037680
dtype: float64


In [117]:
food_dining_df.describe()

,Walk Score,Transit Score,Bike Score,latitude,longitude,Travel_Score
count,571880.000000,558926.000000,571880.000000,571880.000000,571880.000000,558926.000000
mean,51.304052,39.351264,54.378614,33.645455,-93.217919,48.516789
std,17.275925,15.300898,9.470181,4.949490,16.826456,12.947295
min,25.600000,15.100000,29.700000,27.555127,-120.083748,26.733333
25%,37.900000,27.000000,48.900000,28.054921,-110.969799,38.800000
50%,43.700000,33.500000,55.200000,34.397958,-82.788961,45.333333
75%,67.900000,55.200000,61.500000,39.497227,-82.406873,57.500000
max,88.700000,77.100000,72.300000,43.773686,-74.661348,79.366667


In [118]:
shopping_df.describe()

,Walk Score,Transit Score,Bike Score,latitude,longitude,Travel_Score
count,248234.000000,243524.00000,248234.000000,248234.000000,248234.000000,243524.000000
mean,50.051505,38.24327,54.337807,33.804576,-96.475928,47.679551
std,16.858112,14.71468,9.236301,4.790938,17.645846,12.504137
min,25.600000,0.30000,29.700000,27.578648,-120.092141,25.966667
25%,37.300000,25.70000,48.500000,28.101813,-119.631636,38.500000
50%,43.200000,33.50000,55.200000,34.414161,-89.954344,45.000000
75%,65.700000,43.90000,59.600000,39.473746,-82.502675,57.500000
max,88.700000,77.10000,72.300000,43.723526,-74.666433,79.366667


In [119]:
combined[combined['major_category'] == "Automotive"].describe()

,Walk Score,Transit Score,Bike Score,latitude,longitude,Travel_Score
count,134963.000000,133140.000000,134963.000000,134963.000000,134963.000000,133140.000000
mean,49.680337,37.990034,54.328043,33.847750,-96.693056,47.426864
std,16.945173,14.696162,9.014584,4.846440,17.688663,12.432891
min,25.600000,15.100000,29.700000,27.630593,-120.095137,26.733333
25%,35.400000,25.700000,48.500000,28.096844,-119.649916,38.500000
50%,43.000000,33.500000,55.200000,34.412686,-89.929605,45.000000
75%,65.700000,43.900000,59.600000,39.499809,-82.462469,57.500000
max,88.700000,77.100000,72.300000,43.695442,-74.658572,79.366667


In [120]:
combined.describe()

,Walk Score,Transit Score,Bike Score,latitude,longitude,Travel_Score
count,1.904925e+06,1.870270e+06,1.904925e+06,1.904925e+06,1.904925e+06,1.870270e+06
mean,5.038681e+01,3.846400e+01,5.442023e+01,3.373803e+01,-9.622436e+01,4.788925e+01
std,1.691644e+01,1.477670e+01,9.244715e+00,4.799769e+00,1.770312e+01,1.255211e+01
min,2.560000e+01,3.000000e-01,2.970000e+01,2.755513e+01,-1.200951e+02,2.596667e+01
25%,3.790000e+01,2.700000e+01,4.850000e+01,2.808141e+01,-1.196465e+02,3.880000e+01
50%,4.320000e+01,3.350000e+01,5.520000e+01,3.441358e+01,-8.681646e+01,4.500000e+01
75%,6.710000e+01,4.890000e+01,6.150000e+01,3.947329e+01,-8.247620e+01,5.750000e+01
max,8.870000e+01,7.710000e+01,7.230000e+01,4.399390e+01,-7.465857e+01,7.936667e+01


In [121]:
combined

,Place_name,city_x,state,Walk Score,Transit Score,Bike Score,name,city_y,postal_code,latitude,longitude,categories,major_category,Travel_Score
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Women's Clothing,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Accessories,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Children's Clothing,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Men's Clothing,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Adult,Entertainment & Recreation,60.066667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Cafes,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Juice Bars & Smoothies,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Coffee & Tea,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Restaurants,Food & Dining,41.133333


In [122]:
top_10_travel = combined.sort_values(by="Travel_Score")
top_10_travel

,Place_name,city_x,state,Walk Score,Transit Score,Bike Score,name,city_y,postal_code,latitude,longitude,categories,major_category,Travel_Score
354465,"Arlington, TX",Arlington,TX,38.1,0.3,39.5,Residences at Glenview Reserve Apartment Homes,Nashville,37217,36.133400,-86.698535,Home Services,Home & Local Services,25.966667
354464,"Arlington, TX",Arlington,TX,38.1,0.3,39.5,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Home & Garden,Home & Local Services,25.966667
354464,"Arlington, TX",Arlington,TX,38.1,0.3,39.5,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Jewelry,Shopping & Retail,25.966667
354464,"Arlington, TX",Arlington,TX,38.1,0.3,39.5,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Home Decor,Shopping & Retail,25.966667
354464,"Arlington, TX",Arlington,TX,38.1,0.3,39.5,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Outdoor Furniture Stores,Shopping & Retail,25.966667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354476,"Laredo, TX",Laredo,TX,36.8,NaN,39.6,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Jewelry,Shopping & Retail,NaN
354476,"Laredo, TX",Laredo,TX,36.8,NaN,39.6,Wildlife Wonders,Austin,78752,27.839893,-82.764505,Home & Garden,Home & Local Services,NaN
354477,"Laredo, TX",Laredo,TX,36.8,NaN,39.6,Residences at Glenview Reserve Apartment Homes,Nashville,37217,36.133400,-86.698535,Real Estate,Real Estate & Housing,NaN
354477,"Laredo, TX",Laredo,TX,36.8,NaN,39.6,Residences at Glenview Reserve Apartment Homes,Nashville,37217,36.133400,-86.698535,Home Services,Home & Local Services,NaN


In [123]:
combined.groupby(["state", "major_category"]).count()

Place_name  city_x  Walk Score  \
state major_category                                                      
AZ    Automotive                              21217   21217       21217   
      Beauty & Personal Care                  19803   19803       19803   
      Community & Public Services              1169    1169        1169   
      Education                                3045    3045        3045   
      Entertainment & Recreation              22554   22554       22554   
      Events & Weddings                         994     994         994   
      Food & Dining                           65625   65625       65625   
      Health & Medical                        21581   21581       21581   
      Home & Local Services                   34944   34944       34944   
      Other                                    1575    1575        1575   
      Pets                                     6860    6860        6860   
      Professional & Financial Services        5628    5628        5628   
      Real Estate & Housing                    5404    5404        5404   
      Shopping & Retail                       35000   35000       35000   
      Travel & Transportation                  5299    5299        5299   
CA    Automotive                              18208   18208       18208   
      Beauty & Personal Care                  25568   25568       25568   
      Community & Public Services              1376    1376        1376   
      Education                                4288    4288        4288   
      Entertainment & Recreation              39168   39168       39168   
      Events & Weddings                        5104    5104        5104   
      Food & Dining                           69936   69936       69936   
      Health & Medical                        28480   28480       28480   
      Home & Local Services                   39632   39632       39632   
      Other                                    2880    2880        2880   
      Pets                                     6736    6736        6736   
      Professional & Financial Services       11072   11072       11072   
      Real Estate & Housing                    6816    6816        6816   
      Shopping & Retail                       38064   38064       38064   
      Travel & Transportation                 11712   11712       11712   
CO    Entertainment & Recreation                  3       3           3   
      Events & Weddings                           3       3           3   
      Professional & Financial Services           3       3           3   
FL    Automotive                              41268   41268       41268   
      Beauty & Personal Care                  52254   52254       52254   
      Community & Public Services              1878    1878        1878   
      Education                                4950    4950        4950   
      Entertainment & Recreation              59358   59358       59358   
      Events & Weddings                        1860    1860        1860   
      Food & Dining                          192054  192054      192054   
      Health & Medical                        44778   44778       44778   
      Home & Local Services                   64050   64050       64050   
      Other                                    3702    3702        3702   
      Pets                                    16314   16314       16314   
      Professional & Financial Services        9822    9822        9822   
      Real Estate & Housing                    9030    9030        9030   
      Shopping & Retail                       74436   74436       74436   
      Travel & Transportation                 13692   13692       13692   
HI    Beauty & Personal Care                      2       2           2   
      Food & Dining                               2       2           2   
      Health & Medical                            3       3           3   
      Shopping & Retail                           3       3       

In [124]:
yelp_df['state'].value_counts()

state
PA     34039
FL     26330
TN     12056
IN     11247
MO     10913
LA      9924
AZ      9912
NJ      8536
NV      7715
AB      5573
CA      5203
ID      4467
DE      2265
IL      2145
TX         4
CO         3
WA         2
HI         2
MA         2
NC         1
UT         1
MT         1
MI         1
SD         1
XMS        1
VI         1
VT         1
Name: count, dtype: int64

In [125]:
combined['state'].value_counts()

state
FL    589446
CA    309040
AZ    250698
PA    233908
NV    200564
TN     88342
MO     77594
LA     69280
NJ     60668
ID     17334
IL      7819
TX       195
HI        10
CO         9
WA         8
MI         6
MA         4
Name: count, dtype: int64

In [126]:
combined = combined.drop(columns=['categories'])


In [127]:
combined

,Place_name,city_x,state,Walk Score,Transit Score,Bike Score,name,city_y,postal_code,latitude,longitude,major_category,Travel_Score
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Shopping & Retail,60.066667
1,"Los Angeles, CA",Los Angeles,CA,68.6,52.9,58.7,H&M,Santa Barbara,93101,34.420209,-119.700460,Entertainment & Recreation,60.066667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Food & Dining,41.133333
522238,"Boise, ID",Boise City,ID,38.5,22.6,62.3,Dutch Bros Coffee,Boise,83704,43.615401,-116.284689,Food & Dining,41.133333


In [128]:
combined = combined.drop_duplicates()

In [129]:
combined[(combined['state'] == 'CA') & (combined['city_x'] != 'Los Angeles')].head(50)

,Place_name,city_x,state,Walk Score,Transit Score,Bike Score,name,city_y,postal_code,latitude,longitude,major_category,Travel_Score
5204,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,H&M,Santa Barbara,93101,34.420209,-119.700460,Shopping & Retail,44.533333
5204,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,H&M,Santa Barbara,93101,34.420209,-119.700460,Entertainment & Recreation,44.533333
5205,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Helena Avenue Bakery,Santa Barbara,93101,34.414445,-119.690672,Food & Dining,44.533333
5206,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Iron Horse Auto Body,Santa Barbara,93103,34.419620,-119.677032,Automotive,44.533333
5206,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Iron Horse Auto Body,Santa Barbara,93103,34.419620,-119.677032,Shopping & Retail,44.533333
5207,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Tinkle Belle Diaper Service,Santa Barbara,93101,34.420334,-119.710749,Education,44.533333
5207,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Tinkle Belle Diaper Service,Santa Barbara,93101,34.420334,-119.710749,Other,44.533333
5207,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Tinkle Belle Diaper Service,Santa Barbara,93101,34.420334,-119.710749,Shopping & Retail,44.533333
5207,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Tinkle Belle Diaper Service,Santa Barbara,93101,34.420334,-119.710749,Home & Local Services,44.533333
5207,"San Diego, CA",San Diego,CA,53.3,37.3,43.0,Tinkle Belle Diaper Service,Santa Barbara,93101,34.420334,-119.710749,Automotive,44.533333
